# Classical Policy Iteration (Tabular) on Acrobot-v1



## Imports

In [ ]:
import sys, pathlib
import numpy as np
import matplotlib.pyplot as plt

SRC = pathlib.Path.cwd().parents[1] / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from experiment import load_config, build_env, build_agent, train_policy_iteration, make_run_logger
from agents.policy_iteration_agent import TabularPolicyIterationAgent
from analysis.registry import resolve_methods
from analysis.low_rank.rank import row_rank_property_check
from analysis.visualisations.heatmaps import plot_matrix_heatmap

## Reading the config file

Note the non-default filename — this experiment shares the directory with the DQN
baseline (`config.yaml`).

In [ ]:
cfg = load_config("config_classical_pi.yaml")   # loads yaml, resolves device, seeds torch/numpy
print("device:", cfg["experiment"]["_device"])
cfg

## Creating the Environment

`state_observation` makes the observation the native 4-dim `unwrapped.state`, so
`obs_space` below is 4-dim (not the raw 6-dim `cos/sin`) and its bounds are the
finite `±π / ±4π / ±9π` box the grid is laid over.

In [ ]:
env = build_env(cfg)
print("obs_space:", env.observation_space)
print("n_states:", env.get_wrapper_attr("n_states"), "bins:", env.get_wrapper_attr("state_bins"),
      "n_actions:", env.action_space.n)

## Creating the Agent

No network — the agent is the Q table plus the `policy` array (`policy[i]` = action
in bin `i`, kept separate from `Q` so π stays frozen while evaluation overwrites the
table). Grid geometry comes from the env wrapper, so `build_agent` needs only
`agent_cls`.

In [ ]:
agent = build_agent(cfg, env, agent_cls=TabularPolicyIterationAgent)
print("Q table:", agent.Q.shape)

## Run logger

In [ ]:
logger = make_run_logger(cfg, config_path="config_classical_pi.yaml")
if logger:
    print("run artifacts ->", logger.dir)

## Policy iteration

Each iteration: exhaustive MC evaluation (3600 bins × 3 actions = 10 800 generative
rollouts) → greedy improvement → greedy evaluation episodes on fixed seeds. Stops
when the policy is stable, solved (`avg ≥ −100`), or the iteration cap hits. Early
iterations are the slow ones — a weak policy never reaches the goal, so each rollout
runs the full horizon. **`rewards.csv` rows are PI iterations**, not episodes.

With `agent.record_rollouts: true` (in the config), every iteration's generative
rollouts are persisted to `runs/<ts>/mc_rollouts/ep{iter:06d}.npz` — per-rollout
`start_bin / action / return / length / terminated` plus the concatenated per-step
`rewards` and visited `bins`. Step 2 can reload these to recompute Q at any
truncation τ offline (`rewards[:τ]` + `γ**τ · V(bins[τ-1])`) and study the low-rank
completion, without re-running the (expensive) rollouts.

In [ ]:
rewards = train_policy_iteration(cfg, agent, env, run_logger=logger)

## Reward per iteration

In [ ]:
rewards = np.asarray(rewards, dtype=float)
plt.figure(figsize=(8, 4))
plt.plot(rewards, marker="o", label="avg greedy reward")
plt.axhline(cfg["training"]["solved_reward"], ls="--", c="grey", label="solved")
plt.xlabel("PI iteration"); plt.ylabel("avg greedy reward")
plt.title("Classical Policy Iteration on Acrobot-v1")
plt.legend()
if logger:
    plt.savefig(logger.dir / "reward_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## Post-training analysis

`q_matrix_tabular` is the agent's own table — with 3 actions it is rank ≤ 3 by
construction, so the Hankel sweep (ran every iteration during training) is the
meaningful low-rank probe here.

In [ ]:
methods = resolve_methods(cfg["analysis"].get("methods", []) + cfg["analysis"].get("post_methods", []))
for method, names in methods:
    results = method(agent=agent, env=env)
    if not isinstance(results, tuple):
        results = (results,)
    for matrix, name in zip(results, names):
        print(name)
        plot_matrix_heatmap(matrix, name, save_to=logger.figure_path(f"{name} heatmap") if logger else None)
        r, sr, spk, shape, irs, ics, rc, cc, nzr, nzc = row_rank_property_check(matrix, name, save_to=logger.figure_path(name) if logger else None)
        print(f"eff_rank: {r}, stable_rank: {sr:.2f}, spikiness: {spk:.2f}, shape: {shape}, non-zero rows :{nzr}, non-zero cols:{nzc}")
        print(f"top-r leverage spread: row min={irs.min():.4g} max={irs.max():.4g} (uniform {1.0/shape[0]:.4g}) | col min={ics.min():.4g} max={ics.max():.4g} (uniform {1.0/shape[1]:.4g})")
        print(f"coherence score: row={rc:.4g} col={cc:.4g}")

## Greedy rollout video

In [ ]:
from analysis.visualisations.rollout_video import record_greedy_episode
from IPython.display import Video
import glob

video_dir = "videos"
eval_env = build_env(cfg, render_mode="rgb_array")
prefix = record_greedy_episode(agent, eval_env, video_dir, episode=0, seed=cfg["experiment"]["seed"])
mp4 = sorted(glob.glob(f"{video_dir}/{prefix}-*.mp4"))[-1]
print("saved:", mp4)
Video(mp4, embed=True)